# Step 1 · Download the Statistics Canada series

**What this does:** downloads the full history of every Statistics Canada series listed in `../series.csv`
and saves them in one file, `store/statcan.csv`.

**How:** Statistics Canada's Web Data Service (WDS) gives each series a number, its *vector* (e.g. `2062815` is the
seasonally adjusted unemployment rate). We send a list of vectors and get back every month of data for each one.
No API key is needed. All the series take a few seconds.

**Output:** `store/statcan.csv`, one row per series per month:

| id | vector | date | value | release_time |
|---|---|---|---|---|
| unemp_rate | 2062815 | 2026-09-01 | 6.5 | 2026-10-09T08:30 |

The full history is downloaded every time, so revisions (for example the January LFS revisions) come in by themselves.

**To add a series:** add a row to `series.csv` with its vector. To find a vector, open the table on the StatCan website,
choose *Add/Remove data → Customize layout → Display vector identifier and coordinate*.

In [ ]:
import time                # to pause between requests and time the run
import pandas as pd        # tables
import requests            # to call the Statistics Canada website

# The address of the WDS function that returns the latest N data points of a list of vectors
URL = "https://www150.statcan.gc.ca/t1/wds/rest/getDataFromVectorsAndLatestNPeriods"

# How many periods to ask for. 1,000 months is more than 80 years: in practice, the whole history.
LATEST_N = 1000

# How many vectors to send in one request (StatCan accepts long lists; 50 keeps each answer small)
BATCH_SIZE = 50

## 1. Read the list of series

`series.csv` is the registry: every series the page uses, with its source and vector. Here we keep only the
Statistics Canada ones.

In [ ]:
series = pd.read_csv("../series.csv")

# keep the rows whose source is Statistics Canada
statcan = series[series["source"] == "statcan"]

print(len(statcan), "Statistics Canada series to download")
statcan[["id", "table", "vector", "name"]].head()

## 2. Download them

We send the vectors in batches. Each answer has one entry per vector with its list of data points.
Each data point has the month (`refPer`), the value, and the time it was released (`releaseTime`).

StatCan's answers do not always come back in the order we asked, so we match each answer to its series
by the vector number, not by position. If a request fails, we wait a few seconds and try once more.

In [ ]:
# a lookup from vector number to our own short id, e.g. 2062815 -> "unemp_rate"
id_of_vector = dict(zip(statcan["vector"], statcan["id"]))

vectors = list(statcan["vector"])
rows = []                  # every data point of every series ends up in this list
start = time.time()

for i in range(0, len(vectors), BATCH_SIZE):
    batch = vectors[i:i + BATCH_SIZE]
    request = [{"vectorId": int(v), "latestN": LATEST_N} for v in batch]

    # ask, and if it fails, wait 5 seconds and ask once more
    try:
        answer = requests.post(URL, json=request, timeout=120).json()
    except Exception as error:
        print("   request failed, trying again:", error)
        time.sleep(5)
        answer = requests.post(URL, json=request, timeout=120).json()

    # Instead of a list, WDS sometimes sends a message, e.g. {"message": "The product is not released yet"}: a table
    # with a release due that day is locked from midnight until the release (8:30, Ottawa time). Stop and say so.
    if not isinstance(answer, list):
        raise RuntimeError(f"Statistics Canada sent no data for series {i + 1} to {i + len(batch)}: {answer}. "
                           "'Not released yet' means a release is due today: run again after 8:30, Ottawa time.")

    for entry in answer:
        if entry["status"] != "SUCCESS":
            print("   StatCan could not return:", entry)
            continue
        vector = entry["object"]["vectorId"]
        for point in entry["object"]["vectorDataPoint"]:
            rows.append({
                "id": id_of_vector[vector],
                "vector": vector,
                "date": point["refPer"],                 # first day of the month, e.g. 2026-09-01
                "value": point["value"],
                "release_time": point["releaseTime"],    # when StatCan published this number (Ottawa time)
            })

    print(f"   series {i + 1} to {i + len(batch)} done")

data = pd.DataFrame(rows)
print(f"Downloaded {len(data):,} data points in {time.time() - start:.1f} seconds")

## 3. Check that nothing is missing

Every series in the registry must have come back with data. If one is missing, its vector may have been retired
(StatCan does this, rarely, when it reorganises a table). The check stops here rather than publishing a page with a
hole in it.

In [ ]:
missing = sorted(set(statcan["id"]) - set(data["id"]))
if missing:
    raise ValueError(f"No data came back for: {missing}. Check their vectors in series.csv.")

# StatCan marks suppressed or unavailable points with an empty value; drop those
empty = data["value"].isna().sum()
data = data.dropna(subset=["value"])
print("All series came back.", empty, "empty points dropped.")

## 4. Save

One file for all StatCan series. The build notebooks read it from here.

In [ ]:
data = data.sort_values(["id", "date"])
data.to_csv("store/statcan.csv", index=False)

# a short summary: the latest month of each series and when it was released
latest = data.groupby("id").tail(1)[["id", "date", "value", "release_time"]]
print("Saved store/statcan.csv")
print(latest.sort_values("date").to_string(index=False))